# Assignment 2 — Food-101: baseline vs pretrained ResNet-50

- Dataset: Food-101, 101 classes, 75,750 train / 25,250 official test.
- Models: Assignment 1 CNN baseline, ImageNet ResNet-50 full fine-tune, ResNet-50 frozen backbone.
- Seeds: 42, 67, 69; split fixed at seed 42.
- Metrics: accuracy, macro-F1, per-class F1, training time, inference time.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "assignment2").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "assignment2" / "src"))
sys.path.insert(0, str(REPO_ROOT))

import json
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

from training_loop import evaluate, fit, set_seed
from utils.dataloader import build_food101_loaders
from utils.experiment_utils import measure_inference_time, plot_curves, plot_examples
from models.baseline import build_baseline_cnn
from models.resnet import build_resnet50

SEEDS = [42, 67, 69]
SPLIT_SEED = 42
EPOCHS = 30
BATCH_SIZE = 128
WORKERS = 8
LIMIT = None
MODELS = ["baseline", "resnet50_full", "resnet50_frozen"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406]).reshape(1, 1, 3)
IMAGENET_STD = np.array([0.229, 0.224, 0.225]).reshape(1, 1, 3)

RESULTS_DIR = REPO_ROOT / "assignment2" / "results"
CHECKPOINT_DIR = REPO_ROOT / "assignment2" / "checkpoints"
HISTORY_DIR = RESULTS_DIR / "history"
CURVES_DIR = RESULTS_DIR / "curves"
EXAMPLES_DIR = RESULTS_DIR / "examples"
CONFUSION_DIR = RESULTS_DIR / "confusion"

for directory in (RESULTS_DIR, CHECKPOINT_DIR, HISTORY_DIR, CURVES_DIR, EXAMPLES_DIR, CONFUSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)

device: cuda


In [2]:
loaders, class_names = build_food101_loaders(
    128, batch_size=8, workers=0, limit=16, seed=SPLIT_SEED
)
images, labels = next(iter(loaders["train"]))
print("classes:", len(class_names))
print("batch:", tuple(images.shape), images.dtype)
print("labels:", labels[:8].tolist())

classes: 101
batch: (8, 3, 128, 128) torch.float32
labels: [26, 76, 60, 7, 46, 13, 34, 92]


In [3]:
def build_model(name, num_classes, image_size):
    if name == "baseline":
        return build_baseline_cnn(num_classes=num_classes, image_size=image_size)
    if name == "resnet50_full":
        return build_resnet50(num_classes=num_classes, pretrained=True, freeze_backbone=False)
    if name == "resnet50_frozen":
        return build_resnet50(num_classes=num_classes, pretrained=True, freeze_backbone=True)
    raise ValueError(f"unknown model: {name}")

In [4]:
def top_confusions(cm, class_names, k=20):
    pairs = []
    for true_index in range(cm.shape[0]):
        for predicted_index in range(cm.shape[1]):
            if true_index == predicted_index:
                continue
            count = int(cm[true_index, predicted_index])
            if count:
                pairs.append((class_names[true_index], class_names[predicted_index], count))
    return sorted(pairs, key=lambda item: item[2], reverse=True)[:k]


def plot_top_confusions(pairs, path, title):
    if not pairs:
        return
    labels = [f"{true} -> {predicted}" for true, predicted, _ in pairs]
    counts = [count for _, _, count in pairs]

    fig, ax = plt.subplots(figsize=(10, 0.35 * len(pairs)))
    ax.barh(labels[::-1], counts[::-1], color="#0366d6")
    ax.set_xlabel("test images")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def predict(model, loader, device, amp=True):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                logits = model(images)
            predictions.append(logits.argmax(1).cpu())
            targets.append(labels)
    return torch.cat(predictions).numpy(), torch.cat(targets).numpy()

## Controlled experiment: freeze vs full fine-tune

- Hypothesis: full fine-tuning beats a frozen backbone.
- Changed factor: backbone frozen vs fully trainable.
- Fixed factors: data split, augmentation, input size, epochs, optimizer, batch size, seeds.
- Metrics: test accuracy, macro-F1, training time, inference time.


In [5]:
results = []
histories = {}
per_class_metrics = {}
confusion_pairs = {}

for seed in SEEDS:
    for name in MODELS:
        print(f"\n=== {name} | seed {seed} ===")
        set_seed(seed)
        image_size = 128 if name == "baseline" else 224
        loaders, class_names = build_food101_loaders(
            image_size,
            batch_size=BATCH_SIZE,
            workers=WORKERS,
            limit=LIMIT,
            seed=SPLIT_SEED,
        )
        model = build_model(name, len(class_names), image_size).to(DEVICE)
        frozen = name == "resnet50_frozen"
        params = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        checkpoint_path = CHECKPOINT_DIR / f"{name}_seed{seed}.pt"

        history, best_epoch, train_time = fit(
            model,
            loaders,
            DEVICE,
            epochs=EPOCHS,
            lr=1e-4 if name.startswith("resnet") else 1e-3,
            weight_decay=1e-4,
            amp=AMP,
            checkpoint_path=checkpoint_path,
            frozen_backbone=frozen,
            seed=seed,
        )

        test = evaluate(model, loaders["test"], nn.CrossEntropyLoss(), DEVICE, AMP)
        inference_time, inference_ms = measure_inference_time(model, loaders["test"], DEVICE)
        predictions, targets = predict(model, loaders["test"], DEVICE, AMP)

        cm = confusion_matrix(targets, predictions, labels=np.arange(len(class_names)))
        precision, recall, f1, support = precision_recall_fscore_support(
            targets,
            predictions,
            labels=np.arange(len(class_names)),
            average=None,
            zero_division=0,
        )
        pairs = top_confusions(cm, class_names)

        run_name = f"{name}_seed{seed}"
        histories[run_name] = history
        per_class_metrics[run_name] = [
            {
                "class_id": index,
                "class_name": class_names[index],
                "precision": float(precision[index]),
                "recall": float(recall[index]),
                "f1": float(f1[index]),
                "support": int(support[index]),
            }
            for index in range(len(class_names))
        ]
        confusion_pairs[run_name] = [
            {"true": true, "predicted": predicted, "count": count}
            for true, predicted, count in pairs
        ]

        with open(HISTORY_DIR / f"{run_name}.json", "w") as handle:
            json.dump(history, handle, indent=2)
        with open(CONFUSION_DIR / f"{run_name}_top_confusions.json", "w") as handle:
            json.dump(confusion_pairs[run_name], handle, indent=2)
        np.save(CONFUSION_DIR / f"{run_name}.npy", cm)

        plot_curves(history, CURVES_DIR / f"{run_name}.png", run_name)
        plot_top_confusions(pairs, CONFUSION_DIR / f"{run_name}_top_confusions.png", run_name)

        example_images, example_labels = next(iter(loaders["test"]))
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=AMP):
                example_logits = model(example_images.to(DEVICE))
        example_predictions = example_logits.argmax(1).cpu().numpy()
        plot_examples(
            example_images,
            example_labels.numpy(),
            example_predictions,
            class_names,
            EXAMPLES_DIR / f"{run_name}.png",
            run_name,
            IMAGENET_MEAN,
            IMAGENET_STD,
        )

        results.append(
            {
                "model": name,
                "seed": seed,
                "params": params,
                "trainable_params": trainable,
                "image_size": image_size,
                "best_epoch": best_epoch,
                "train_time_s": train_time,
                "inference_time_s": inference_time,
                "inference_ms_per_sample": inference_ms,
                "val_accuracy": history["validation_accuracy"][best_epoch - 1],
                "val_macro_f1": history["validation_macro_f1"][best_epoch - 1],
                "test_accuracy": test["accuracy"],
                "test_macro_f1": test["macro_f1"],
            }
        )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df


=== baseline | seed 42 ===


epoch 01/30 | train loss 4.2091 acc 0.0663 | val loss 3.8797 acc 0.1100 macro-F1 0.0878 | 32.9s


epoch 02/30 | train loss 3.7268 acc 0.1391 | val loss 3.5614 acc 0.1646 macro-F1 0.1393 | 32.0s


epoch 03/30 | train loss 3.4180 acc 0.1938 | val loss 3.3553 acc 0.2038 macro-F1 0.1841 | 30.4s


epoch 04/30 | train loss 3.2091 acc 0.2320 | val loss 3.1648 acc 0.2384 macro-F1 0.2190 | 30.3s


...

epoch 27/30 | train loss 1.2863 acc 0.6861 | val loss 1.4235 acc 0.6444 macro-F1 0.6399 | 38.1s


epoch 28/30 | train loss 1.2742 acc 0.6888 | val loss 1.4144 acc 0.6474 macro-F1 0.6435 | 56.6s


epoch 29/30 | train loss 1.2629 acc 0.6896 | val loss 1.4064 acc 0.6477 macro-F1 0.6433 | 41.4s


epoch 30/30 | train loss 1.2502 acc 0.6911 | val loss 1.3990 acc 0.6484 macro-F1 0.6445 | 35.4s


,model,seed,params,trainable_params,image_size,best_epoch,train_time_s,inference_time_s,inference_ms_per_sample,val_accuracy,val_macro_f1,test_accuracy,test_macro_f1
0,baseline,42,8829349,8829349,128,26,907.341204,11.048900,0.437580,0.342442,0.335627,0.375525,0.369793
1,resnet50_full,42,23714981,23714981,224,4,1975.356891,15.463274,0.612407,0.804224,0.803177,0.840950,0.840223
2,resnet50_frozen,42,23714981,206949,224,30,1327.229693,15.001953,0.594137,0.649241,0.645334,0.693545,0.690529
3,baseline,67,8829349,8829349,128,25,911.756279,10.536974,0.417306,0.369109,0.364824,0.408079,0.404377
4,resnet50_full,67,23714981,23714981,224,9,1601.728559,16.903400,0.669442,0.804884,0.805273,0.842812,0.843165
5,resnet50_frozen,67,23714981,206949,224,30,1264.166588,20.652482,0.817920,0.650561,0.646340,0.692515,0.689113
6,baseline,69,8829349,8829349,128,30,2208.284280,9.917316,0.392765,0.381518,0.376326,0.421901,0.417937
7,resnet50_full,69,23714981,23714981,224,4,1517.238745,14.796757,0.586010,0.803300,0.802964,0.842218,0.842302
8,resnet50_frozen,69,23714981,206949,224,30,1164.539487,19.675198,0.779216,0.648449,0.644474,0.692673,0.689221


In [6]:
summary_spec = {
    "runs": ("seed", "count"),
    "test_accuracy_mean": ("test_accuracy", "mean"),
    "test_accuracy_std": ("test_accuracy", "std"),
    "test_macro_f1_mean": ("test_macro_f1", "mean"),
    "test_macro_f1_std": ("test_macro_f1", "std"),
    "val_accuracy_mean": ("val_accuracy", "mean"),
    "val_macro_f1_mean": ("val_macro_f1", "mean"),
    "best_epoch_mean": ("best_epoch", "mean"),
    "train_time_s_mean": ("train_time_s", "mean"),
    "inference_ms_per_sample_mean": ("inference_ms_per_sample", "mean"),
}

summary_df = results_df.groupby("model").agg(**summary_spec).reset_index()
summary_df

,model,runs,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,val_accuracy_mean,val_macro_f1_mean,best_epoch_mean,train_time_s_mean,inference_ms_per_sample_mean
0,baseline,3,0.401835,0.023810,0.397369,0.024825,0.364356,0.358926,27.000000,1342.460588,0.415884
1,resnet50_frozen,3,0.692911,0.000554,0.689621,0.000788,0.649417,0.645383,30.000000,1251.978589,0.730424
2,resnet50_full,3,0.841993,0.000951,0.841897,0.001512,0.804136,0.803805,5.666667,1698.108065,0.622620


In [7]:
ab = summary_df.set_index("model").loc[["resnet50_frozen", "resnet50_full"]]

ablation_summary = {
    "changed_factor": "frozen backbone vs full fine-tune",
    "hypothesis": "full fine-tuning beats a frozen backbone on Food-101",
    "seeds": SEEDS,
    "frozen_test_accuracy_mean": float(ab.loc["resnet50_frozen", "test_accuracy_mean"]),
    "full_test_accuracy_mean": float(ab.loc["resnet50_full", "test_accuracy_mean"]),
    "delta_test_accuracy": float(ab.loc["resnet50_full", "test_accuracy_mean"] - ab.loc["resnet50_frozen", "test_accuracy_mean"]),
    "frozen_test_macro_f1_mean": float(ab.loc["resnet50_frozen", "test_macro_f1_mean"]),
    "full_test_macro_f1_mean": float(ab.loc["resnet50_full", "test_macro_f1_mean"]),
    "delta_test_macro_f1": float(ab.loc["resnet50_full", "test_macro_f1_mean"] - ab.loc["resnet50_frozen", "test_macro_f1_mean"]),
    "decision_criterion": "compare the mean over the three seeds and the spread across seeds",
}
ablation_summary

{'changed_factor': 'frozen backbone vs full fine-tune',
 'hypothesis': 'full fine-tuning beats a frozen backbone on Food-101',
 'seeds': [42, 67, 69],
 'frozen_test_accuracy_mean': 0.692910891089109,
 'full_test_accuracy_mean': 0.841993399339934,
 'delta_test_accuracy': 0.14908250825082503,
 'frozen_test_macro_f1_mean': 0.6896207264202534,
 'full_test_macro_f1_mean': 0.8418965881373488,
 'delta_test_macro_f1': 0.1522758617170954,
 'decision_criterion': 'compare the mean over the three seeds and the spread across seeds'}

In [8]:
config = {
    "dataset": "food-101",
    "split_seed": SPLIT_SEED,
    "training_seeds": SEEDS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "limit": LIMIT,
    "amp": AMP,
    "monitor": "validation_macro_f1",
    "models": MODELS,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

payload = {
    "config": config,
    "results": results,
    "summary": summary_df.to_dict(orient="records"),
    "ablation": ablation_summary,
    "history": histories,
    "per_class": per_class_metrics,
    "top_confusions": confusion_pairs,
}

results_df.to_csv(RESULTS_DIR / "results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)

with open(RESULTS_DIR / "results.json", "w") as handle:
    json.dump(payload, handle, indent=2)

print("saved", RESULTS_DIR / "results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment2/results/results.json


## Next steps

- Write the final A2 report, slides, and YouTube video.
- Link the final deliverables from `assignment2/index.html` and the landing page.
